# Bespoke Labs Nimble 9B Decision Model for Argument Stance Classification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dustinober1/Arg_Me/blob/main/notebooks/06_nimble_9b_decision_model_stance.ipynb)

### What is Nimble?
[Nimble](https://ollama.com/library/nimble) is a **9B decision model** developed by Bespoke Labs, fine-tuned from Qwen 9B. It is specifically designed for high-capacity, typed classification tasks.

### Why Nimble 9B for Debate Stance Detection?
1. **Trained on Contrastive Pairs**: Bespoke Labs trains Nimble on paired contrastive examples where flipping a single factual assertion inverts the correct decision. This makes Nimble exceptionally sensitive to subtle dialectic markers, concessions, and double negations.
2. **Larger 9B Parameter Capacity**: Scaling from Tev1's 4B parameters to Nimble's 9B parameters provides greater capacity to navigate argumentative complexity without free-form generation latency.
3. **Zero-Token Output Decoding**: Like Tev1, Nimble uses Ollama's `/v1/systemone` endpoint to score choice candidate options directly in $O(1)$ decoding steps.

In [ ]:
# 1. Environment Detection and Dependency Setup
import sys
import os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
if IN_COLAB:
    print("Provisioning dependencies inside Google Colab runtime...")
    !pip install -q datasets huggingface_hub pandas pyarrow scikit-learn requests matplotlib seaborn tqdm
    if not os.path.exists('Arg_Me') and not os.path.exists('src'):
        print("Cloning Arg_Me repository into Colab...")
        !git clone https://github.com/dustinober1/Arg_Me.git
        %cd Arg_Me
    elif os.path.exists('Arg_Me') and not os.path.exists('src'):
        %cd Arg_Me
    if '/content/Arg_Me' not in sys.path:
        sys.path.insert(0, '/content/Arg_Me')
    print('Colab environment configured! Current working directory:', os.getcwd())
else:
    print("Running in local environment.")

In [ ]:
# 2. Imports and Ollama Connectivity Verification
import time
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix,
)

# Ensure project root in sys.path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

from src.dataset_loader import load_sample_df

# Configure Ollama URL
OLLAMA_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434").rstrip("/")

try:
    res = requests.get(f"{OLLAMA_URL}/api/tags", timeout=3.0)
    models = [m['name'] for m in res.json().get('models', [])]
    print(f"Connected to Ollama at {OLLAMA_URL}!")
    print("Available models:", models)
    has_nimble = any("nimble" in m for m in models)
    if has_nimble:
        print("Successfully verified 'nimble' is pulled and ready.")
    else:
        print("Warning: 'nimble' was not found in tags. Run: ollama pull nimble")
except Exception as e:
    print(f"Could not connect to Ollama at {OLLAMA_URL}: {e}")

In [ ]:
# 3. Nimble Decision Helper Function
def query_nimble_decision(
    conclusion: str,
    premise_text: str,
    model_name: str = "nimble",
    base_url: str = OLLAMA_URL,
    timeout: float = 30.0,
) -> dict:
    """
    Query Ollama's /v1/systemone endpoint using Bespoke Labs' Nimble 9B decision model.
    """
    payload = {
        "model": model_name,
        "state": f"Topic: {conclusion}\nPremise: {premise_text}",
        "questions": {
            "stance": {
                "type": "choice",
                "instructions": "What is the argumentative stance of the premise with respect to the topic?",
                "criteria": {
                    "PRO": "The premise supports, defends, or argues in favor of the topic/resolution.",
                    "CON": "The premise opposes, attacks, refutes, or argues against the topic/resolution.",
                },
            }
        },
    }
    
    t0 = time.time()
    try:
        resp = requests.post(f"{base_url}/v1/systemone", json=payload, timeout=timeout)
        latency = (time.time() - t0) * 1000.0
        if resp.status_code == 200:
            data = resp.json()
            ans = data.get("answers", {}).get("stance", {})
            probs = ans.get("probabilities", {})
            return {
                "choice": ans.get("choice", "UNKNOWN"),
                "pro_prob": float(probs.get("PRO", 0.5)),
                "con_prob": float(probs.get("CON", 0.5)),
                "confidence": float(ans.get("confidence", 0.0)),
                "latency_ms": latency,
                "error": None,
            }
        else:
            return {"choice": "UNKNOWN", "pro_prob": 0.5, "con_prob": 0.5, "confidence": 0.0, "latency_ms": latency, "error": f"HTTP {resp.status_code}"}
    except Exception as e:
        return {"choice": "UNKNOWN", "pro_prob": 0.5, "con_prob": 0.5, "confidence": 0.0, "latency_ms": (time.time() - t0) * 1000.0, "error": str(e)}

# Test on a sample argument
sample_out = query_nimble_decision(
    conclusion="Renewable Energy Subsidies Should Be Phased Out",
    premise_text="Subsidies distort free market pricing, prop up inefficient firms, and create long-term taxpayer liabilities.",
)
print("Sample Decision on Nimble 9B:")
print(sample_out)

In [ ]:
# 4. Load Dataset and Configure Evaluation Slice
data_path = Path("../data/args_me_benchmark_5k.parquet")
if not data_path.exists():
    data_path = Path("data/args_me_benchmark_5k.parquet")

if data_path.exists():
    df = pd.read_parquet(data_path)
else:
    print("Loading streaming dataset slice...")
    df = load_sample_df(n_records=5000, save_parquet=True, parquet_filename="args_me_benchmark_5k.parquet")

df_clean = df[df["stance"].isin(["PRO", "CON"])].copy().reset_index(drop=True)

# Held-out 20% test split
_, test_df = train_test_split(
    df_clean, test_size=0.20, random_state=42, stratify=df_clean["stance"]
)
print(f"Held-out test set size: {len(test_df)}")

# Set desired evaluation sample size (adjust as you like!)
SAMPLE_SIZE = 50
eval_slice = test_df.sample(SAMPLE_SIZE, random_state=42).copy().reset_index(drop=True)
print(f"Selected evaluation slice: N = {len(eval_slice)}")

In [ ]:
# 5. Execute Evaluation Loop on Nimble 9B
print(f"Running Nimble 9B decision evaluation on {len(eval_slice)} test arguments...")

preds = []
pro_probs = []
confidences = []
latencies = []

for _, row in tqdm(eval_slice.iterrows(), total=len(eval_slice), desc="Evaluating Nimble 9B"):
    res = query_nimble_decision(str(row["conclusion"]), str(row["premise_text"]))
    preds.append(res["choice"])
    pro_probs.append(res["pro_prob"])
    confidences.append(res["confidence"])
    latencies.append(res["latency_ms"])

eval_slice["pred_stance"] = preds
eval_slice["pro_prob"] = pro_probs
eval_slice["confidence"] = confidences
eval_slice["latency_ms"] = latencies
eval_slice["is_correct"] = eval_slice["stance"] == eval_slice["pred_stance"]

y_true = eval_slice["stance"].values
y_pred = np.array(preds)
pro_probs_arr = np.array(pro_probs)

acc = accuracy_score(y_true, y_pred)
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
p_c, r_c, f1_c, _ = precision_recall_fscore_support(y_true, y_pred, labels=["CON", "PRO"], zero_division=0)
auc = roc_auc_score(y_true == "PRO", pro_probs_arr)
cm = confusion_matrix(y_true, y_pred, labels=["CON", "PRO"])

print("\n=== BESPOKE LABS NIMBLE 9B BENCHMARK RESULTS ===")
print(f"Accuracy:         {acc:.4f} ({acc * 100:.2f}%)")
print(f"Macro F1:         {f1:.4f}")
print(f"CON F1:           {f1_c[0]:.4f} | PRO F1: {f1_c[1]:.4f}")
print(f"ROC-AUC:          {auc:.4f}")
print(f"Average Latency:  {np.mean(latencies):.1f} ms per decision")
print(f"Confusion Matrix:\n{cm}")

In [ ]:
# 6. Cross-Paradigm Benchmark Comparison Chart
benchmark_comparison = [
    {"Model": "Majority Baseline (Zero-R)", "Accuracy": 0.5020, "Type": "Baseline"},
    {"Model": "VADER Sentiment Polarity", "Accuracy": 0.5040, "Type": "Affect"},
    {"Model": "Linear SVM (Contextual TF-IDF)", "Accuracy": 0.5070, "Type": "Traditional ML"},
    {"Model": "Logistic Regression (Premise TF-IDF)", "Accuracy": 0.5490, "Type": "Traditional ML"},
    {"Model": "Zero-Shot NLI DeBERTa-v3 Small", "Accuracy": 0.5230, "Type": "Transformer"},
    {"Model": "Fine-Tuned RoBERTa-base Cross-Encoder", "Accuracy": 0.5890, "Type": "Transformer"},
    {"Model": "Together AI Tev1:4B Decision Model", "Accuracy": 0.7500, "Type": "Decision 4B"},
    {"Model": "Bespoke Labs Nimble:9B Decision Model", "Accuracy": acc, "Type": "Decision 9B"},
]
plot_df = pd.DataFrame(benchmark_comparison).sort_values("Accuracy", ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))
palette = {
    "Baseline": "#94a3b8",
    "Affect": "#94a3b8",
    "Traditional ML": "#3b82f6",
    "Transformer": "#10b981",
    "Decision 4B": "#8b5cf6",
    "Decision 9B": "#ec4899",
}
colors = [palette[t] for t in plot_df["Type"]]
bars = ax.barh(plot_df["Model"], plot_df["Accuracy"], color=colors, height=0.6)
ax.axvline(0.50, color="#ef4444", linestyle="--", linewidth=1.5, label="Chance Baseline (50%)")

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.005, bar.get_y() + bar.get_height()/2, f"{w:.1%}", va="center", fontsize=11, fontweight="bold")

ax.set_xlim(0.40, 0.95)
ax.set_xlabel("Accuracy", fontsize=12)
ax.set_title("Stance Classification Performance Across Architectural Eras (Up to 9B Decision Models)", fontsize=14, fontweight="bold", pad=12)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# 7. Qualitative Error Review: Inspect Sample Predictions
preview_cols = ["conclusion", "premise_text", "stance", "pred_stance", "pro_prob", "confidence", "is_correct"]
eval_slice[preview_cols].head(10)